# TabPFN from scratch

Which of our preprocessing steps does TabPFN actually need, and which of its settings help? Every experiment uses the same 5 folds (the folds of `scripts/cv.py`) and reports R² on **all validation rows** and on **inlier validation rows** (the test set looks outlier-free, so the inlier score is the one that matches Kaggle). Needs a GPU.

R² is a scaled MSE, so the Bayes-optimal point prediction is the **mean** of TabPFN's predicted distribution (its default). A custom loss only helps for other metrics; median/mode are tested only as a check of the predicted distribution.

In [ ]:
import time

import numpy as np
import pandas as pd

from sklearn.metrics import r2_score
from sklearn.model_selection import RepeatedKFold
from tabpfn import TabPFNRegressor

from aml.config import CV_FOLDS
from aml.config import OUTLIER_FRAC
from aml.config import SEED
from aml.data import load_clean_data
from aml.data import load_data
from aml.features import correlated_block
from aml.features import outlier_scores


X_raw, y, X_test_raw = load_data()  # all 832 features, nothing removed
X_clean, _, X_test_clean, inlier = load_clean_data()  # 726 cleaned features
block = set(correlated_block(pd.concat([X_clean, X_test_clean], ignore_index=True)))
X_block = X_clean[[c for c in X_clean.columns if c in block]]
outlier_rank = outlier_scores(X_clean, y, seed=SEED).index  # most anomalous first
FOLDS = list(
    RepeatedKFold(n_splits=CV_FOLDS, n_repeats=1, random_state=SEED).split(X_raw)
)
print(
    X_raw.shape, X_clean.shape, X_block.shape, f"{(~inlier).sum()} flagged outlier rows"
)

RESULTS = []


def fold_r2(pred, rows):
    """Mean R² over the folds, on the given validation rows."""
    return np.mean(
        [r2_score(y.iloc[va[rows[va]]], pred[va[rows[va]]]) for _, va in FOLDS]
    )


def cv(name, X, train_mask=None, output_types=("mean",), seeds=(SEED,), **params):
    """5-fold CV of TabPFN. train_mask: rows allowed in training (None = all).
    Several output_types come from one predict call; several seeds are averaged."""
    start = time.time()
    oof = {t: np.zeros(len(X)) for t in output_types}
    for tr, va in FOLDS:
        tr = tr if train_mask is None else tr[train_mask[tr]]
        for seed in seeds:
            model = TabPFNRegressor(device="auto", random_state=seed, **params)
            model.fit(X.iloc[tr], y.iloc[tr])
            out = model.predict(X.iloc[va], output_type="main")
            for t in output_types:
                oof[t][va] += out[t] / len(seeds)
    all_rows = np.ones(len(X), dtype=bool)
    for t in output_types:
        label = name if len(output_types) == 1 else f"{name} [{t}]"
        row = {
            "experiment": label,
            "R² all rows": fold_r2(oof[t], all_rows),
            "R² inlier rows": fold_r2(oof[t], inlier),
            "minutes": (time.time() - start) / 60,
        }
        RESULTS.append(row)
        print(
            f"{label:45s} all {row['R² all rows']:.4f} | inliers {row['R² inlier rows']:.4f}"
        )
    return oof

## A–D: which preprocessing does TabPFN need?

A: purely raw. B: raw, training without the flagged outlier rows. C: our cleaning (726 features, copies merged) without outliers. D: the correlated block (185) without outliers (the current setup, 0.756 before).

In [ ]:
_ = cv("A raw, all rows", X_raw)
_ = cv("B raw, outliers removed", X_raw, train_mask=inlier)
_ = cv("C cleaned 726, outliers removed", X_clean, train_mask=inlier)
_ = cv("D block 185, outliers removed", X_block, train_mask=inlier)

## Settings

Set `X_best` / `TRAIN_BEST` to the winner of A–D before running the cells below.

In [ ]:
X_best, TRAIN_BEST = X_block, inlier  # <- the winner of A-D

In [ ]:
# E: how many outlier rows to drop (0 = keep all)
for frac in (0.0, 0.02, OUTLIER_FRAC, 0.08):
    keep = ~X_best.index.isin(outlier_rank[: round(frac * len(X_best))])
    _ = cv(f"E outlier share {frac:.1%}", X_best, train_mask=keep)

In [ ]:
# F: size of TabPFN's internal ensemble
for n in (4, 8, 16, 32):
    _ = cv(f"F n_estimators={n}", X_best, train_mask=TRAIN_BEST, n_estimators=n)

In [ ]:
# G: mean (R²-optimal) vs median vs mode of the predicted distribution
_ = cv(
    "G point prediction",
    X_best,
    train_mask=TRAIN_BEST,
    output_types=("mean", "median", "mode"),
)

In [ ]:
# H: average over 5 random seeds
_ = cv("H 5 seeds averaged", X_best, train_mask=TRAIN_BEST, seeds=range(5))

In [ ]:
# I: TabPFN's own outlier handling: soft-clips feature values beyond mean +/- n std
# (log-squashed, two-pass robust mean/std), per feature. Off by default for
# regression. It clips extreme values, it does not remove rows.
for n_std in (4.0, 12.0):
    for label, mask in (("all rows", None), ("outlier rows removed", TRAIN_BEST)):
        _ = cv(
            f"I clip at {n_std:g} std, {label}",
            X_best,
            train_mask=mask,
            inference_config={"OUTLIER_REMOVAL_STD": n_std},
        )

In [ ]:
results = pd.DataFrame(RESULTS).set_index("experiment").round(4)
results.to_csv("../outputs/tabpfn_experiments.csv")
results